### Materials Project

In [1]:
import json

with open('..\..\dataset\materials_project\mp.2019.04.01.json', 'r') as f:
    materials_project = json.load(f)

In [26]:
len(materials_project)

133420

In [27]:
materials_project[0]

{'material_id': 'mp-1103373',
 'graph': {'atom': [19, 19, 19, 19, 50, 50, 50, 50, 50, 50, 50, 50],
  'bond': [4.0822286454580965,
   4.073241967223304,
   4.082228645458096,
   3.6292457761945593,
   3.920221924173147,
   3.9096256691695745,
   3.920532967367759,
   3.8031390835701937,
   3.8009379795496954,
   3.809838507634042,
   3.8101585620156038,
   3.800774412133431,
   3.8032962427312538,
   3.6589146410395696,
   3.6544307101235862,
   3.658751280070727,
   4.082228645458096,
   4.073241967223304,
   4.0822286454580965,
   3.6292457761945602,
   3.920221924173146,
   3.9096256691695745,
   3.9205329673677594,
   3.6587512800707263,
   3.6544307101235862,
   3.6589146410395696,
   3.8007744121334315,
   3.8032962427312533,
   3.8098385076340415,
   3.810158562015604,
   3.8031390835701937,
   3.8009379795496954,
   3.6292457761945602,
   4.0822286454580965,
   4.073241967223304,
   4.0822286454580965,
   3.9202219241731466,
   3.9096256691695745,
   3.920532967367759,
   3.6587

In [28]:
materials_project[0].keys(), (len(materials_project[0].keys()))

(dict_keys(['material_id', 'graph', 'formation_energy_per_atom', 'structure']),
 4)

In [2]:
materials_project[0].get('structure')

"# generated using pymatgen\ndata_KSn2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.68165201\n_cell_length_b   6.68165201\n_cell_length_c   9.89046224\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   119.82109815\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   KSn2\n_chemical_formula_sum   'K4 Sn8'\n_cell_volume   383.08489264\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  K  K1  1  0.666721  0.333279  0.566528  1\n  K  K2  1  0.333279  0.666721  0.433472  1\n  K  K3  1  0.333279  0.666721  0.066528  1\n  K  K4  1  0.666721  0.333279  0.933472  1\n  Sn  Sn5  1  -0.000000  -0.000000  0.500000  1\n  Sn  Sn6  1  -0.000000  -0.000000  -0.000000  1\n  Sn  Sn7  1  0.169826  0.340009  0.75000

### Extract Materials Project to CSV and Graph.pt

In [1]:
import json
import warnings
from pathlib import Path
from collections import Counter
import pandas as pd
import torch
from torch_geometric.data import Data
from pymatgen.core import Structure, Composition
from pymatgen.core.periodic_table import Element

In [2]:
RAW_PATH = Path("..\..\dataset\materials_project\mp.2019.04.01.json")
OUT_DIR = Path(".\data\processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUT_DIR / "materials.csv"
GRAPH_PT_PATH = OUT_DIR / "materials_graphs.pt"

with open(RAW_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

In [3]:
def clean_value(value):
    if value in [None, "", "na", "NA", "N/A", "null"]:
        return None
    return value


def parse_cif_safely(cif_text):
    warning_messages = []

    if not isinstance(cif_text, str) or not cif_text.strip():
        return None, warning_messages, "Empty CIF string"

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", UserWarning)
            structure = Structure.from_str(cif_text, fmt="cif")

            warning_messages = [str(w.message) for w in caught]

        return structure, warning_messages, None

    except Exception as e:
        return None, warning_messages, str(e)


def formula_from_graph_atom(atom_numbers):
    """
    Recover formula from graph.atom if CIF parsing fails.
    Example:
    [19, 19, 19, 19, 50, 50, ...] -> KSn2
    """
    if not isinstance(atom_numbers, list) or len(atom_numbers) == 0:
        return None, []

    try:
        counts = Counter(atom_numbers)

        comp_dict = {}
        elements = []

        for atomic_number, count in counts.items():
            symbol = Element.from_Z(int(atomic_number)).symbol
            comp_dict[symbol] = int(count)
            elements.append(symbol)

        formula = Composition(comp_dict).reduced_formula
        return formula, sorted(elements)

    except Exception:
        return None, []


def make_pyg_graph(record, uid, graph_index):
    """
    Convert Materials Project prebuilt graph into PyTorch Geometric Data object.
    """
    graph = record.get("graph", {})

    atom = graph.get("atom", [])
    bond = graph.get("bond", [])
    index1 = graph.get("index1", [])
    index2 = graph.get("index2", [])

    if not atom or not bond or not index1 or not index2:
        return None, "Missing graph fields"

    if not (len(bond) == len(index1) == len(index2)):
        return None, "Graph edge lengths do not match"

    formation_energy = clean_value(record.get("formation_energy_per_atom"))

    if formation_energy is None:
        return None, "Missing formation_energy_per_atom"

    x = torch.tensor(atom, dtype=torch.long).view(-1, 1)

    edge_index = torch.tensor(
        [index1, index2],
        dtype=torch.long
    )

    edge_attr = torch.tensor(
        bond,
        dtype=torch.float
    ).view(-1, 1)

    y = torch.tensor(
        [float(formation_energy)],
        dtype=torch.float
    )

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y
    )

    data.material_uid = uid
    data.graph_index = graph_index

    return data, None

In [4]:
materials_rows = []
pyg_graphs = []
skipped_rows = []

counter = 1
num_rows = len(raw_data)

for record in raw_data:
    print(f"Record {counter}/{num_rows} processing...")
    source_id = clean_value(record.get("material_id"))

    if source_id is None:
        skipped_rows.append({
            "source_id": None,
            "reason": "Missing material_id"
        })
        continue

    uid = f"MP_{source_id}"

    graph = record.get("graph", {})
    atom_numbers = graph.get("atom", [])
    num_atoms_graph = len(atom_numbers)
    num_edges = len(graph.get("bond", []))

    # ---------- Parse CIF structure ----------
    cif_text = record.get("structure")
    structure, cif_warnings, cif_error = parse_cif_safely(cif_text)

    formula = None
    element_symbols = []
    lattice_a = lattice_b = lattice_c = None
    alpha = beta = gamma = None
    volume = None
    density = None
    num_atoms_structure = None

    if structure is not None:
        composition = structure.composition
        lattice = structure.lattice

        formula = composition.reduced_formula
        element_symbols = [str(el) for el in composition.elements]

        lattice_a = lattice.a
        lattice_b = lattice.b
        lattice_c = lattice.c
        alpha = lattice.alpha
        beta = lattice.beta
        gamma = lattice.gamma
        volume = lattice.volume
        density = structure.density
        num_atoms_structure = len(structure)

    else:
        # fallback: recover formula and elements from graph.atom
        formula, element_symbols = formula_from_graph_atom(atom_numbers)

    # If formula is still missing, skip this material
    if formula is None:
        skipped_rows.append({
            "source_id": source_id,
            "reason": "Could not recover formula"
        })
        continue

    # ---------- Create PyG graph ----------
    graph_index = len(pyg_graphs)

    pyg_data, graph_error = make_pyg_graph(
        record=record,
        uid=uid,
        graph_index=graph_index
    )

    if pyg_data is None:
        skipped_rows.append({
            "source_id": source_id,
            "reason": graph_error
        })
        continue

    pyg_graphs.append(pyg_data)

    formation_energy = clean_value(record.get("formation_energy_per_atom"))

    materials_rows.append({
        "uid": uid,
        "source_id": source_id,
        "source": "Materials Project",

        "formula": formula,
        "elements_str": ";".join(element_symbols),

        "formation_energy_per_atom": formation_energy,

        "num_atoms": num_atoms_structure or num_atoms_graph,
        "num_edges": num_edges,

        "lattice_a": lattice_a,
        "lattice_b": lattice_b,
        "lattice_c": lattice_c,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "volume": volume,
        "density": density,

        # Important: this links CSV row to the graph list in .pt
        "graph_index": graph_index,

        # Useful for debugging
        "cif_parse_warning": " | ".join(cif_warnings) if cif_warnings else None,
        "cif_parse_error": cif_error
    })
    counter += 1


materials_df = pd.DataFrame(materials_rows)
materials_df.to_csv(CSV_PATH, index=False)
torch.save(pyg_graphs, GRAPH_PT_PATH)


print("Done.")
print(f"Saved CSV: {CSV_PATH}")
print(f"Saved graph dataset: {GRAPH_PT_PATH}")
print(f"Materials exported: {len(materials_df)}")
print(f"Graphs exported: {len(pyg_graphs)}")

print(f"Skipped materials: {len(skipped_rows)}")
for row in skipped_rows:
    print(row)

Record 1/133420 processing...
Record 2/133420 processing...
Record 3/133420 processing...
Record 4/133420 processing...
Record 5/133420 processing...
Record 6/133420 processing...
Record 7/133420 processing...
Record 8/133420 processing...
Record 9/133420 processing...
Record 10/133420 processing...
Record 11/133420 processing...
Record 12/133420 processing...
Record 13/133420 processing...
Record 14/133420 processing...
Record 15/133420 processing...
Record 16/133420 processing...
Record 17/133420 processing...
Record 18/133420 processing...
Record 19/133420 processing...
Record 20/133420 processing...
Record 21/133420 processing...
Record 22/133420 processing...
Record 23/133420 processing...
Record 24/133420 processing...
Record 25/133420 processing...
Record 26/133420 processing...
Record 27/133420 processing...
Record 28/133420 processing...
Record 29/133420 processing...
Record 30/133420 processing...
Record 31/133420 processing...
Record 32/133420 processing...
Record 33/133420 

d:\AI_PROJECTS\ThanhHoa_AIContest\.conda\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


Record 11550/133420 processing...
Record 11551/133420 processing...
Record 11552/133420 processing...
Record 11553/133420 processing...
Record 11554/133420 processing...
Record 11555/133420 processing...
Record 11556/133420 processing...
Record 11557/133420 processing...
Record 11558/133420 processing...
Record 11559/133420 processing...
Record 11560/133420 processing...
Record 11561/133420 processing...
Record 11562/133420 processing...
Record 11563/133420 processing...
Record 11564/133420 processing...
Record 11565/133420 processing...
Record 11566/133420 processing...
Record 11567/133420 processing...
Record 11568/133420 processing...
Record 11569/133420 processing...
Record 11570/133420 processing...
Record 11571/133420 processing...
Record 11572/133420 processing...
Record 11573/133420 processing...
Record 11574/133420 processing...
Record 11575/133420 processing...
Record 11576/133420 processing...
Record 11577/133420 processing...
Record 11578/133420 processing...
Record 11579/1

d:\AI_PROJECTS\ThanhHoa_AIContest\.conda\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


Record 69565/133420 processing...
Record 69566/133420 processing...
Record 69567/133420 processing...
Record 69568/133420 processing...
Record 69569/133420 processing...
Record 69570/133420 processing...
Record 69571/133420 processing...
Record 69572/133420 processing...
Record 69573/133420 processing...
Record 69574/133420 processing...
Record 69575/133420 processing...
Record 69576/133420 processing...
Record 69577/133420 processing...
Record 69578/133420 processing...
Record 69579/133420 processing...
Record 69580/133420 processing...
Record 69581/133420 processing...
Record 69582/133420 processing...
Record 69583/133420 processing...
Record 69584/133420 processing...
Record 69585/133420 processing...
Record 69586/133420 processing...
Record 69587/133420 processing...
Record 69588/133420 processing...
Record 69589/133420 processing...
Record 69590/133420 processing...
Record 69591/133420 processing...
Record 69592/133420 processing...
Record 69593/133420 processing...
Record 69594/1

In [5]:
materials_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 133420 entries, 0 to 133419
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   uid                        133420 non-null  str    
 1   source_id                  133420 non-null  str    
 2   source                     133420 non-null  str    
 3   formula                    133420 non-null  str    
 4   elements_str               133420 non-null  str    
 5   formation_energy_per_atom  133420 non-null  float64
 6   num_atoms                  133420 non-null  int64  
 7   num_edges                  133420 non-null  int64  
 8   lattice_a                  133420 non-null  float64
 9   lattice_b                  133420 non-null  float64
 10  lattice_c                  133420 non-null  float64
 11  alpha                      133420 non-null  float64
 12  beta                       133420 non-null  float64
 13  gamma                      133420 non-nu

In [ ]:
import torch
import pandas as pd

In [7]:
graph_path = "data/processed/materials_graphs.pt"
csv_path = "data/processed/materials.csv"

graphs = torch.load(graph_path, weights_only=False)
materials_df = pd.read_csv(csv_path)

In [4]:
len(graphs)

133420

In [5]:
g = graphs[0]
print(g)
print("x:", g.x.shape)
print("edge_index:", g.edge_index.shape)
print("edge_attr:", g.edge_attr.shape)
print("y:", g.y)
print("material_uid:", g.material_uid)
print("graph_index:", g.graph_index)

Data(x=[12, 1], edge_index=[2, 164], edge_attr=[164, 1], y=[1], material_uid='MP_mp-1103373', graph_index=0)
x: torch.Size([12, 1])
edge_index: torch.Size([2, 164])
edge_attr: torch.Size([164, 1])
y: tensor([-0.2449])
material_uid: MP_mp-1103373
graph_index: 0


In [8]:
for i in range(5):
    row = materials_df.iloc[i]
    graph_index = int(row["graph_index"])
    graph = graphs[graph_index]

    print(row["uid"], graph.material_uid, graph_index)

MP_mp-1103373 MP_mp-1103373 0
MP_mp-1106279 MP_mp-1106279 1
MP_mp-1224519 MP_mp-1224519 2
MP_mp-1204424 MP_mp-1204424 3
MP_mp-1226074 MP_mp-1226074 4
